# Notebook 6: AI Infrastructure Agent & Root Cause Analysis (RCA)

**GPU Fleet Autopilot — Research & Simulation Suite**

This notebook explores the **AI Infrastructure Agent's deterministic rule engine** and its **10 diagnostic tools** for automated triage and root-cause determination on degraded GPU clusters.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sample_telemetry.csv")

def evaluate_agent_rule(row):
    if row["dcgm_ecc_dbe_volatile_total"] > 0 or row["dcgm_xid_errors"] == 62:
        return "HARDWARE_DBE_FAULT", ["quarantine_gpu", "run_diagnostics", "request_rma"]
    elif row["dcgm_gpu_temp"] >= 92.0 and row["dcgm_clock_throttle_reasons"] != 0:
        return "THERMAL_RUNAWAY", ["quarantine_gpu", "run_diagnostics"]
    elif row["dcgm_xid_errors"] == 79:
        return "GPU_FALLEN_OFF_BUS", ["restart_driver", "quarantine_gpu"]
    elif row["dcgm_nvlink_error_count"] > 30 or row["dcgm_xid_errors"] == 92:
        return "NVLINK_CRC_STORM", ["quarantine_gpu", "run_diagnostics"]
    elif row["performance_ratio"] < 0.70:
        return "PERFORMANCE_STRAGGLER", ["quarantine_gpu"]
    return "HEALTHY_OR_MONITORING", []

diagnoses = [evaluate_agent_rule(r) for _, r in df.iterrows()]
df["agent_diagnosis"] = [d[0] for d in diagnoses]

print("Agent rule evaluation breakdown across fleet:")
print(df["agent_diagnosis"].value_counts())